In [7]:
import os
import time
import requests

API_KEY = "llx-UZyoip0e7FS8a4hvDMx0oH4saPtPOpiK2kiANw8dSktFpYea"
pdf_files = ["final_cutline.pdf", "document_cutline.pdf"]
output_filename = "parsed_result.md"

headers = {
    "Authorization": f"Bearer {API_KEY}",
    "Accept": "application/json"
}

print("🚀 LlamaParse 직접 통신(REST API) 모드 시작...")

with open(output_filename, "w", encoding="utf-8") as f:
    for file in pdf_files:
        print(f"📄 '{file}' 서버로 전송 중...")
        
        # 1. 파일 업로드 및 분석 요청
        with open(file, "rb") as pdf:
            files = {"file": (file, pdf, "application/pdf")}
            data = {"language": "ko"}
            upload_res = requests.post(
                "https://api.cloud.llamaindex.ai/api/parsing/upload",
                headers=headers,
                files=files,
                data=data
            )
            
        if upload_res.status_code != 200:
            print(f"❌ 업로드 실패: {upload_res.text}")
            continue
            
        job_id = upload_res.json()["id"]
        print(f"⏳ 전송 완료. 서버에서 AI가 문서를 분석 중입니다 (약 30초~1분 소요)...")

        # 2. 서버 분석이 끝날 때까지 3초마다 확인 (Polling)
        while True:
            status_res = requests.get(f"https://api.cloud.llamaindex.ai/api/parsing/job/{job_id}", headers=headers)
            status = status_res.json()["status"]
            
            if status == "SUCCESS":
                print(f"✅ 분석 완료! 결과 다운로드 중...")
                break
            elif status == "ERROR":
                print(f"❌ 분석 실패: {status_res.text}")
                break
                
            time.sleep(3) # 3초 대기 후 다시 상태 확인

        # 3. 마크다운 텍스트 결과 받아와서 파일에 쓰기
        if status == "SUCCESS":
            result_res = requests.get(f"https://api.cloud.llamaindex.ai/api/parsing/job/{job_id}/result/markdown", headers=headers)
            markdown_text = result_res.json()["markdown"]
            f.write(markdown_text + "\n\n")
            print(f"🎉 '{file}' 마크다운 저장 완료!\n")

print(f"🎊 모든 작업이 완료되어 '{output_filename}'에 저장되었습니다.")

🚀 LlamaParse 직접 통신(REST API) 모드 시작...
📄 'final_cutline.pdf' 서버로 전송 중...
⏳ 전송 완료. 서버에서 AI가 문서를 분석 중입니다 (약 30초~1분 소요)...
✅ 분석 완료! 결과 다운로드 중...
🎉 'final_cutline.pdf' 마크다운 저장 완료!

📄 'document_cutline.pdf' 서버로 전송 중...
⏳ 전송 완료. 서버에서 AI가 문서를 분석 중입니다 (약 30초~1분 소요)...
✅ 분석 완료! 결과 다운로드 중...
🎉 'document_cutline.pdf' 마크다운 저장 완료!

🎊 모든 작업이 완료되어 'parsed_result.md'에 저장되었습니다.


In [8]:
import pandas as pd

In [9]:
with open("parsed_result.md", "r", encoding="utf-8") as f:
    lines = f.readlines()

table_rows = []

In [10]:
for line in lines:
    line = line.strip()
    if line.count("|") >= 3 and '---' not in line:  # 테이블 행으로 보이는 라인
        cells = [cell.strip() for cell in line.split("|")] # 셀 단위로 분리하고 양쪽 공백 제거
        clean_row = [cell for cell in cells if cell != '']  # 빈 셀 제거

        if len(clean_row) >= 4: # 최소 4개 이상의 셀이 있는 행만 추가
            table_rows.append(clean_row)

In [11]:
new_data_df = pd.DataFrame(table_rows) # 추출된 테이블 데이터를 DataFrame으로 변환

print(f"총 {len(new_data_df)}줄의 표 데이터를 추출했습니다.")
display(new_data_df.head(10)) # 추출된 데이터의 상위 10줄을 출력하여 확인

총 3238줄의 표 데이터를 추출했습니다.


,0,1,2,3,4,5,6,7,8,9,10,11,12
0,단지명,공급유형,공급대상(신청자격),구분,당첨자 합격선 (㎡),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,디에이치 포레센트,행복주택(고령자),우선,우선 1순위,6점(자치구 전입일 : 1987.03.20),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,디에이치 포레센트,행복주택( ),우선,우선 1순위,6점(자치구 전입일 : 1996.01.09),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,르엘대치 (대치제 2 지구),행복주택(고령자),우선,우선 1순위,5점,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,르엘대치 (대치제 2 지구),행복주택(고령자),일반,무작위 추첨,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,르엘대치 (대치제 2 지구),행복주택(신혼부부계층),우선,우선 1순위,6점(자치구 전입일 : 2016.10.26),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,보라매자이 (동작트인시아),행복주택(고령자),우선,우선 1순위,6점(자치구 전입일 : 1998.09.14),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,보라매자이 (동작트인시아),행복주택(고령자),일반,무작위 추첨,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,보라매자이 (동작트인시아),행복주택(신혼부부계층),우선,우선 1순위,6점(자치구 전입일 : 2017.11.16),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,보라매자이 (동작트인시아),행복주택(신혼부부계층),일반,일반 1순위 중 추첨,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [12]:
import numpy as np

In [14]:
new_data_df = new_data_df.iloc[:, :5].copy() # 첫 5개 열만 남기고 나머지는 제거 (필요에 따라 조정 가능)

In [15]:
# 데이터프레임의 열 이름을 지정 (예시로 5개의 열 이름을 사용)
new_data_df.columns = ['단지명', '공급유형', '신청자격', '구분', '결과텍스트']

In [16]:
df_clean = new_data_df[new_data_df['단지명'] != '단지명'].copy() # '단지명'이 포함된 행 제거

In [17]:
# '결과텍스트'에서 점수 추출하여 새로운 열로 추가
df_clean['커트라인점수'] = df_clean['결과텍스트'].astype(str).str.extract(r'(\d+)점').astype(float) 

In [18]:
df_clean = df_clean.dropna(subset=['커트라인점수']).copy() # 커트라인점수가 없는 행 제거

In [19]:
# '당첨순위' 열 추가, 순위가 없는 경우 1로 간주
df_clean['당첨순위'] = df_clean['결과텍스트'].astype(str).str.extract(r'(\d+)순위').astype(float).fillna(1.0).astype(int)

In [20]:
# 신청자격 및 공급 유형 글자 다듬기
def clean_eligibility(text):
    for keyword in ['청년', '신혼부부', '고령자', '대학생', '주거급여수급자']:
        if keyword in text:
            return keyword
        return '일반가구'

df_clean['신청자격'] = df_clean['신청자격'].apply(clean_eligibility)
df_clean['공급유형'] = df_clean['공급유형'].astype(str).str.replace('형', '') + '형' # 공급유형 뒤에 '형' 붙이기
df_clean['심사단계'] = '최종' # 심사단계 열 추가, 모든 행에 '최종'으로 설정   

In [21]:
# 기존 데이터와 완벽히 동일한 6개 컬럼만 남기기
final_new_df = df_clean[['단지명', '공급유형', '신청자격', '당첨순위', '심사단계', '커트라인점수']]

In [22]:
print(f"정제 완료. 점수제 데이터 총 {len(final_new_df)}줄을 확보.")
display(final_new_df.head(10)) # 최종 정제된 데이터의 상위 10줄을 출력하여 확인

정제 완료. 점수제 데이터 총 282줄을 확보.


,단지명,공급유형,신청자격,당첨순위,심사단계,커트라인점수
1,디에이치 포레센트,행복주택(고령자)형,일반가구,1,최종,6.0
2,디에이치 포레센트,행복주택( )형,일반가구,1,최종,6.0
3,르엘대치 (대치제 2 지구),행복주택(고령자)형,일반가구,1,최종,5.0
5,르엘대치 (대치제 2 지구),행복주택(신혼부부계층)형,일반가구,1,최종,6.0
6,보라매자이 (동작트인시아),행복주택(고령자)형,일반가구,1,최종,6.0
8,보라매자이 (동작트인시아),행복주택(신혼부부계층)형,일반가구,1,최종,6.0
10,보라매자이 (동작트인시아),행복주택(고령자)형,일반가구,1,최종,6.0
12,보라매자이 (동작트인시아),행복주택(신혼부부계층)형,일반가구,1,최종,6.0
28,44,행복주택(신혼부부계층)형,일반가구,1,최종,6.0
29,59,행복주택(신혼부부계층)형,일반가구,1,최종,6.0


In [57]:
import pandas as pd
import numpy as np
import re
import joblib
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import r2_score, mean_absolute_error
from xgboost import XGBRegressor

In [58]:
df = pd.read_csv('cutline_data.csv')

In [59]:
# 임대 유형 구분(공고연도 유무로 판별)
df['임대유형'] = df['공고연도'].apply(
    lambda x: '행복주택' if (pd.isna(x) or str(x).strip() == '') else '청년안심주택'
)
print(f"청년안심주택: {(df['임대유형']=='청년안심주택').sum()}행")
print(f"행복주택:    {(df['임대유형']=='행복주택').sum()}행")

청년안심주택: 394행
행복주택:    1016행


In [60]:
# 행복주택 노이즈 행 제거
# 단지병이 파싱 오류인 행 필터링(숫자만, 너무짧음, 의미없는 키워드)
NOISE = {'우선', '일반', '신혼부부', '고령자', '대학생',
        '주거급여수급자', '주거급여 수급자', '신혼부부계층', '고령자계층'}

def is_valid_name(row):
    if row['임대유형'] != '행복주택':
        return True
    name = str(row['단지명']).strip()
    if name in NOISE:
        return False
    if re.match(r'^\d+$', name):   # 숫자만 있는 경우
        return False
    if len(name) <= 2:              # 너무 짧은 경우
        return False
    return True

df = df[df.apply(is_valid_name, axis=1)].reset_index(drop=True)
print(f"\n노이즈 제거 후 전체: {len(df)}행")
print(f"청년안심주택: {(df['임대유형']=='청년안심주택').sum()}행")
print(f"행복주택: {(df['임대유형']=='행복주택').sum()}행")


노이즈 제거 후 전체: 1278행
청년안심주택: 394행
행복주택: 884행


In [61]:
# 신청자격 정규화
# 청년안심주택: 신혼Ⅰ/Ⅱ, 청년(셰어) 등 다양한 표기 → 통일
def normalize_elig(x):
    x = str(x).strip()
    if '대학' in x:
        return '대학생'
    if '주거급여' in x or '수급' in x:
        return '주거급여수급자'
    if '고령' in x:
        return '고령자'
    if '신혼' in x and ('Ⅱ' in x or 'II' in x):
        return '신혼부부II'
    if '신혼' in x:
        return '신혼부부I'
    if '청년' in x or '셰어' in x:
        return '청년'
    return '일반가구'

df['신청자격_clean'] = df['신청자격'].apply(normalize_elig)

In [62]:
# Feature 생성
# 공급유형 숫자 추출
df['공급유형_숫자'] = df['공급유형'].apply(
    lambda x: float(re.sub(r'[^0-9.]', '', str(x))) if re.sub(r'[^0-9.]', '', str(x)) else 39.0
)

In [63]:
# 신청자격 인코딩
le_elig = LabelEncoder()
le_elig.fit(['청년', '신혼부부I', '신혼부부II', '고령자',
            '주거급여수급자', '대학생', '일반가구'])
df['신청자격_encoded'] = le_elig.transform(df['신청자격_clean'])

In [64]:
# 심사단계 인코딩
le_stage = LabelEncoder()
le_stage.fit(['서류', '최종'])
df['심사단계_encoded'] = le_stage.transform(
    df['심사단계'].apply(lambda x: x if x in ['서류', '최종'] else '최종')
)

In [65]:
# 임대유형 인코딩
le_type = LabelEncoder()
le_type.fit(['청년안심주택', '행복주택'])
df['임대유형_encoded'] = le_type.transform(df['임대유형'])

In [66]:
# 타겟 인코딩 (임대유형 내에서만 평균 계산)
global_mean = df['커트라인점수'].mean()
target_map = {}
for (rent_type, name), grp in df.groupby(['임대유형', '단지명']):
    target_map[f'{rent_type}|{name}'] = grp['커트라인점수'].mean()

df['단지명_encoded'] = df.apply(
    lambda r: target_map.get(f"{r['임대유형']}|{r['단지명']}", global_mean), axis=1
)

In [67]:
# 학습
X = df[['임대유형_encoded', '단지명_encoded', '공급유형_숫자',
        '신청자격_encoded', '당첨순위', '심사단계_encoded']]
y = df['커트라인점수']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
# LightGBM 제거, XGBoost 단독
xgb_model = XGBRegressor(
    n_estimators=200,
    learning_rate=0.05,
    max_depth=4,       # 과적합 방지
    random_state=42,
    verbosity=0
)
xgb_model.fit(X_train, y_train)

y_pred = xgb_model.predict(X_test)
r2  = r2_score(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)

print(f"\nR2 점수: {r2:.4f}")
print(f"MAE:    ±{mae:.2f}점")


R2 점수: 0.5887
MAE:    ±0.86점


In [68]:
# feature 중요도 출력
feat_names = ['임대유형', '단지명(인코딩)', '공급유형(숫자)',
            '신청자격', '당첨순위', '심사단계']
importances = xgb_model.feature_importances_
for name, imp in sorted(zip(feat_names, importances), key=lambda x: -x[1]):
    print(f"  {name}: {imp:.3f}")

  단지명(인코딩): 0.336
  신청자격: 0.239
  당첨순위: 0.170
  임대유형: 0.161
  공급유형(숫자): 0.063
  심사단계: 0.031


In [69]:
# 저장
joblib.dump(xgb_model, 'cutline_model.pkl')
joblib.dump(target_map, 'target_encoding_map.pkl')
joblib.dump(le_type,  'le_type.pkl')
joblib.dump(le_elig,  'le_elig.pkl')
joblib.dump(le_stage, 'le_stage.pkl')
print("\n모델 및 인코더 저장 완료")


모델 및 인코더 저장 완료


In [70]:
import pandas as pd
import numpy as np
import re
import joblib
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import r2_score, mean_absolute_error
from xgboost import XGBRegressor

df = pd.read_csv('cutline_data.csv')

# ── 임대유형 구분 ──
df['임대유형'] = df['공고연도'].apply(
    lambda x: '행복주택' if (pd.isna(x) or str(x).strip() == '') else '청년안심주택'
)

# ── 노이즈 제거 ──
NOISE = {'우선', '일반', '신혼부부', '고령자', '대학생',
        '주거급여수급자', '주거급여 수급자'}

def is_valid(row):
    if row['임대유형'] != '행복주택':
        return True
    name = str(row['단지명']).strip()
    return name not in NOISE and not re.match(r'^\d+$', name) and len(name) > 2

df = df[df.apply(is_valid, axis=1)].reset_index(drop=True)

# ── 공통 전처리 ──
def normalize_elig(x):
    x = str(x).strip()
    if '대학' in x: return '대학생'
    if '주거급여' in x or '수급' in x: return '주거급여수급자'
    if '고령' in x: return '고령자'
    if '신혼' in x and ('Ⅱ' in x or 'II' in x): return '신혼부부II'
    if '신혼' in x: return '신혼부부I'
    if '청년' in x or '셰어' in x: return '청년'
    return '일반가구'

df['신청자격_clean'] = df['신청자격'].apply(normalize_elig)
df['공급유형_숫자'] = df['공급유형'].apply(
    lambda x: float(re.sub(r'[^0-9.]', '', str(x))) if re.sub(r'[^0-9.]', '', str(x)) else 39.0
)

le_elig = LabelEncoder()
le_elig.fit(['청년', '신혼부부I', '신혼부부II', '고령자',
            '주거급여수급자', '대학생', '일반가구'])
df['신청자격_encoded'] = le_elig.transform(df['신청자격_clean'])

le_stage = LabelEncoder()
le_stage.fit(['서류', '최종'])
df['심사단계_encoded'] = le_stage.transform(
    df['심사단계'].apply(lambda x: x if x in ['서류', '최종'] else '최종')
)

# ── 유형별 타겟 인코딩 ──
global_mean = df['커트라인점수'].mean()
target_map = {}
for (rtype, name), grp in df.groupby(['임대유형', '단지명']):
    target_map[f'{rtype}|{name}'] = grp['커트라인점수'].mean()

df['단지명_encoded'] = df.apply(
    lambda r: target_map.get(f"{r['임대유형']}|{r['단지명']}", global_mean), axis=1
)

# ── 공고연도 추가 (청년안심주택만 유효) ──
df['공고연도_clean'] = df['공고연도'].fillna(2023.0).astype(float)


# ════════════════════════════════════
# 유형별 완전 분리 학습
# ════════════════════════════════════

results = {}

for rent_type in ['청년안심주택', '행복주택']:
    sub = df[df['임대유형'] == rent_type].copy()
    print(f"\n{'='*40}")
    print(f"{rent_type} 학습 ({len(sub)}행)")

    if rent_type == '청년안심주택':
        # 청년안심주택: 공고연도 포함
        features = ['단지명_encoded', '공급유형_숫자', '신청자격_encoded',
                    '당첨순위', '심사단계_encoded', '공고연도_clean']
    else:
        # 행복주택: 공고연도 없음
        features = ['단지명_encoded', '공급유형_숫자', '신청자격_encoded',
                    '당첨순위', '심사단계_encoded']

    X = sub[features]
    y = sub['커트라인점수']

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42
    )

    model = XGBRegressor(
        n_estimators=200,
        learning_rate=0.05,
        max_depth=4,
        random_state=42,
        verbosity=0
    )
    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)
    r2  = r2_score(y_test, y_pred)
    mae = mean_absolute_error(y_test, y_pred)

    print(f"R2:  {r2:.4f}")
    print(f"MAE: ±{mae:.2f}점")

    # feature 중요도
    feat_imp = sorted(zip(features, model.feature_importances_),
                    key=lambda x: -x[1])
    for name, imp in feat_imp:
        print(f"  {name}: {imp:.3f}")

    results[rent_type] = model

# ── 저장 ──
joblib.dump(results['청년안심주택'], 'model_youth_safe.pkl')
joblib.dump(results['행복주택'],     'model_happy.pkl')
joblib.dump(target_map, 'target_encoding_map.pkl')
joblib.dump(le_elig,    'le_elig.pkl')
joblib.dump(le_stage,   'le_stage.pkl')

print("\n✅ 유형별 모델 저장 완료")
print("  model_youth_safe.pkl  ← 청년안심주택")
print("  model_happy.pkl       ← 행복주택")


청년안심주택 학습 (394행)
R2:  0.6193
MAE: ±1.41점
  신청자격_encoded: 0.874
  단지명_encoded: 0.038
  공고연도_clean: 0.035
  당첨순위: 0.025
  공급유형_숫자: 0.018
  심사단계_encoded: 0.010

행복주택 학습 (885행)
R2:  0.4189
MAE: ±0.78점
  신청자격_encoded: 0.425
  단지명_encoded: 0.344
  공급유형_숫자: 0.138
  당첨순위: 0.093
  심사단계_encoded: 0.000

✅ 유형별 모델 저장 완료
  model_youth_safe.pkl  ← 청년안심주택
  model_happy.pkl       ← 행복주택
